In [0]:
# --- CRIANDO A TABELA DE FUNCIONÁRIOS (Tabela da Esquerda) ---

# 1. Criamos uma lista contendo as tuplas de dados (cada parêntese representa uma pessoa)
emp_data = [
    (1, "Alice", "New York"),
    (2, "Bob", "San Francisco"),
    (3, "Charlie", "Chicago"),
    (4, "David", "Boston"),
    (5, "Emma", "Seattle")
]

# 2. Definimos os nomes das colunas nessa mesma ordem
emp_schema = ["emp_id", "name", "city"]

# 3. O Spark transforma esses dados em um DataFrame (uma tabela distribuída)
df_emp = spark.createDataFrame(emp_data, emp_schema)

# --- CRIANDO A TABELA DE SALÁRIOS (Tabela da Direita) ---

# 4. Criamos a lista com os salários e departamentos
sal_data = [
    (1, 70000, "HR"),
    (2, 90000, "Engineering"),
    (3, 80000, "Sales"),
    (6, 85000, "Marketing"),
    (7, 95000, "Finance")
]

# 5. Definimos os nomes das colunas da segunda tabela
sal_schema = ["emp_id", "salary", "department"]

# 6. Criamos o DataFrame de salários
df_sal = spark.createDataFrame(sal_data, sal_schema)

# --- PERMITINDO CONSULTAS EM SQL NATIVO ---

# 7. Registramos ambas as tabelas no catálogo temporário do Spark para podermos usar %sql
df_emp.createOrReplaceTempView("employee")
df_sal.createOrReplaceTempView("salary")

In [0]:
%sql
SELECT * FROM employee;
SELECT * FROM salary;

In [0]:
# df_emp -> Tabela da Esquerda
# .join() -> Comando para cruzar tabelas
# df_sal -> Tabela da Direita
# on="emp_id" -> Coluna usada para fazer o cruzamento
# how="inner" -> Tipo de junção: traz APENAS correspondências perfeitas
df_inner = df_emp.join(df_sal, on="emp_id", how="inner")

display(df_inner)

display(df_inner)

In [0]:
%sql
-- Em SQL nativo:
SELECT e.emp_id, e.name, e.city, s.salary, s.department
FROM employee e
INNER JOIN salary s ON e.emp_id = s.emp_id;

In [0]:
# how="outer" -> Diz ao Spark: "Não jogue nada fora! Se faltar dado em um dos lados, coloque null"
df_outer = df_emp.join(df_sal, on="emp_id", how="outer")

display(df_outer)

In [0]:
%sql
SELECT e.emp_id AS emp_id_emp, s.emp_id AS emp_id_sal, e.name, e.city, s.salary, s.department
FROM employee e
FULL OUTER JOIN salary s ON e.emp_id = s.emp_id;

In [0]:
# --- Célula 4 (PySpark) ---
# Mantém todas as linhas da tabela da esquerda (df_emp)
df_left = df_emp.join(df_sal, on="emp_id", how="left") # ou how="left_outer"

display(df_left)

In [0]:
%sql
SELECT e.emp_id, e.name, e.city, s.salary, s.department
FROM employee e
LEFT OUTER JOIN salary s ON e.emp_id = s.emp_id;

In [0]:
# how="right" -> Preserva TODOS os registros da tabela da direita (df_sal)
df_right = df_emp.join(df_sal, on="emp_id", how="right")

display(df_right)

In [0]:
%sql
SELECT s.emp_id, e.name, e.city, s.salary, s.department
FROM employee e
RIGHT OUTER JOIN salary s ON e.emp_id = s.emp_id;

In [0]:
# how="left_semi" -> Filtra df_emp mantendo apenas quem existe em df_sal (sem trazer colunas de df_sal)
df_semi = df_emp.join(df_sal, on="emp_id", how="left_semi")

display(df_semi)

In [0]:
%sql
SELECT * FROM employee e
LEFT SEMI JOIN salary s ON e.emp_id = s.emp_id;

In [0]:
# how="left_anti" -> Traz apenas os funcionários da esquerda que NÃO possuem salário cadastrado
df_anti = df_emp.join(df_sal, on="emp_id", how="left_anti")

display(df_anti)

In [0]:
%sql
SELECT * FROM employee e
LEFT ANTI JOIN salary s ON e.emp_id = s.emp_id;

In [0]:
# crossJoin() não precisa do parâmetro 'on' porque ele cruza todas as linhas com todas
df_cross = df_emp.crossJoin(df_sal)

display(df_cross)

In [0]:
%sql
SELECT * 
FROM employee 
CROSS JOIN salary;